[Accueil](../../../index.ipynb) > [Sommaire de Terminale](../../index.ipynb)

# TP IMDb

[IMDb](https://www.imdb.com/fr/) (Internet Movie Database) est une base de donnée en ligne comportant des films, des shows TV, jeux vidéos...
Leurs bases sont librement téléchargeables.

Le but de ce TP est d'installer la base sur un serveur SQL dédié.

Connectez vous en ssh (en root) sur le serveur sql-server : mot passe sql_@dmin


## Installation et configuration de MariaDB

### Installation

[MariaDB](https://mariadb.org/) est un système de gestion de base de données relationnelle (SGBDR) open source sous licence GPLv2, créé en 2009 comme un fork de MySQL par Michael « Monty » Widenius.

Sur le sql-server nous allons installer mariaDB

```bash
apt update
apt install mariadb-server mariadb-client -y
```
Ces deux paquets ont des rôles distincts :
- mariadb-server : installe le serveur de bases de données.
- mariadb-client : fournit les outils en ligne de commande pour s'y connecter.

Vérifier ensuite que le service est actif :

```bash
systemctl status mariadb
```

Activer également le démarrage automatique :

```bash
systemctl enable mariadb
```

### Première connexion
MariaDB utilise initialement l'authentification système pour permettre à l'administrateur Linux root de se connecter localement.

```bash
mariadb
```

A l'invite de commande de mariaDB on peut faire notre première requête:

```sql
SELECT VERSION()
```


```sql
SHOW DATABASES;
```

Pour quitter:

```sql
exit;
```

Exécuter la commande suivante:

```bash
ss -lntp | grep 3306
```
*ss* permet d'afficher des informations sur les sockets réseau du système Linux. Elle est notamment utilisée pour savoir quels services écoutent sur le réseau et sur quels ports.

Chaque option a une signification :

<table>
<tr>
    <th>Option</th><th>Signification</th><th>Rôle</th>
</tr>
<tr>
    <td>-l</td><td>listening</td><td>Affiche uniquement les sockets en écoute</td>
</tr>
<tr>
    <td>-n</td><td>numeric</td><td>Affiche les adresses IP et les ports sous forme numérique</td>
</tr>
<tr>
    <td>-t</td><td>TCP</td><td>Affiche uniquement les connexions TCP</td>
</tr>
<tr>
    <td>-p</td><td>processes/td><td>Affiche le processus associé à chaque socket</td>
</tr>
</table>

```
LISTEN 0      80         127.0.0.1:3306      0.0.0.0:*    users:(("mariadbd",pid=12221,fd=20))
```

- LISTEN : Le serveur attend des connexions entrantes
- 0 : Nombre de connexions en attente dans la file d'acceptation
- 80 : Taille maximale de cette file
- 127.0.0.1 : Adresse IP sur laquelle le serveur écoute
- 3306 : Port TCP utilisé par MariaDB
- 0.0.0.0:* : Adresse et port des clients distants, sans restriction particulière dans cette colonne
- mariadbd : Processus responsable de l'écoute
- pid=12221 : Identifiant du processus Linux


127.0.0.1 correspond à l'adresse de boucle locale (localhost). Cela signifie que MariaDB accepte actuellement les connexions uniquement depuis le conteneur lui-même.

Depuis **un terminal de votre poste**, vérifiez si vous pouvez atteindre le port en question:

```
nc -vz 192.168.1.141 3306
```

Nous allons configurer MariaDB pour qu'il écoute sur l'adresse IP du conteneur. (Adresse à déterminer)

```
nano /etc/mysql/mariadb.conf.d/50-server.cnf
```

Il faut modifier bind-address = 127.0.0.1 par celle de l'ip du serveur.

On redémarre le serveur

```
systemctl restart mariadb
```

Et on vérifie que la modification a été prise en compte.

Maintenant les machines du réseau peuvent se connecter à la base mariadb.

<div class="alert alert-warning">
Cela ne signifie pas encore que n'importe quel utilisateur peut se connecter : MariaDB distingue l'adresse d'écoute réseau et les autorisations accordées aux comptes SQL. Nous allons justement configurer ces dernières ensuite.
</div>

Depuis **un terminal de votre poste**, vérifiez si, maintenant, vous pouvez atteindre le port en question:


```
nc -vz 192.168.1.141 3306
```








## Installation  et importation des données IMDb

Créer un repétoire dédié:

```
mkdir -p /srv/imdb/download
cd /srv/imdb/download
```

Installons les outils nécessaires:

```
apt install -y wget gzip
```
### Téléchargement des 4 fichiers

```
wget -c https://datasets.imdbws.com/title.basics.tsv.gz
wget -c https://datasets.imdbws.com/name.basics.tsv.gz
wget -c https://datasets.imdbws.com/title.principals.tsv.gz
wget -c https://datasets.imdbws.com/title.ratings.tsv.gz
```

- L'option -c permet de reprendre un téléchargement interrompu.
- Les fichiers proviennent directement du serveur officiel IMDb : datasets.imdbws.com.
- Le téléchargement peut prendre un certain temps, particulièrement pour title.principals.tsv.gz.

### Vérification

```
ls -lh
```
Puis vérifier l'intégrité des archives :

```
for fichier in *.tsv.gz; do
    echo "Vérification de $fichier"
    gzip -t "$fichier" && echo "OK"
done
```

Cette vérification décompresse les données en mémoire pour vérifier leur intégrité, sans créer les fichiers TSV décompressés.


### Examiner les fichiers

Les fichiers sont au format TSV, nous allons lire uniquement les premières lignes des 4 fichiers:

```
for fichier in *.tsv.gz; do
    echo
    echo "===== $fichier ====="
    zcat "$fichier" | head -n 2
done
```

Maintenant on peut passer à la conception du shéma relationnel.

On peut partir sur les types suivants:

- Table title_basics
<table>
  <tr><th>Colonne</th><th>Type SQL</th><th>Rôle</th></tr>
  <tr><td>tconst</td><td>VARCHAR(12)</td><td>Identifiant de l'œuvre, clé primaire</td></tr>
  <tr><td>titleType</td><td>VARCHAR(20)</td><td>Type d'œuvre</td></tr>
  <tr><td>primaryTitle</td><td>VARCHAR(512)</td><td>Titre principal</td></tr>
  <tr><td>originalTitle</td><td>VARCHAR(512)</td><td>Titre original</td></tr>
  <tr><td>isAdult</td><td>TINYINT</td><td>Contenu adulte ou non</td></tr>
  <tr><td>startYear</td><td>SMALLINT</td><td>Année de début</td></tr>
  <tr><td>endYear</td><td>SMALLINT</td><td>Année de fin</td></tr>
  <tr><td>runtimeMinutes</td><td>INT</td><td>Durée en minutes</td></tr>
  <tr><td>genres</td><td>VARCHAR(128)</td><td>Genres</td></tr>
</table>

- Table name_basics
<table>
  <tr><th>Colonne</th><th>Type SQL</th><th>Rôle</th></tr>
  <tr><td>nconst</td><td>VARCHAR(12)</td><td>Identifiant de la personne, clé primaire</td></tr>
  <tr><td>primaryName</td><td>VARCHAR(256)</td><td>Nom</td></tr>
  <tr><td>birthYear</td><td>SMALLINT</td><td>Année de naissance</td></tr>
  <tr><td>deathYear</td><td>SMALLINT</td><td>Année de décès</td></tr>
  <tr><td>primaryProfession</td><td>VARCHAR(255)</td><td>Professions</td></tr>
  <tr><td>knownForTitles</td><td>TEXT</td><td>Œuvres connues</td></tr>
</table>

- Table title_principals
<table>
  <tr><th>Colonne</th><th>Type SQL</th><th>Rôle</th></tr>
  <tr><td>tconst</td><td>VARCHAR(12)</td><td>Identifiant de l'œuvre</td></tr>
  <tr><td>ordering</td><td>SMALLINT</td><td>Ordre de participation</td></tr>
  <tr><td>nconst</td><td>VARCHAR(12)</td><td>Identifiant de la personne</td></tr>
  <tr><td>category</td><td>VARCHAR(32)</td><td>Catégorie de participation</td></tr>
  <tr><td>job</td><td>VARCHAR(512)</td><td>Fonction</td></tr>
  <tr><td>characters</td><td>TEXT</td><td>Personnages interprétés</td></tr>
</table>
<p>La clé primaire sera composée de tconst et ordering.</p>

- Table title_ratings
<table>
  <caption>T</caption>
  <tr><th>Colonne</th><th>Type SQL</th><th>Rôle</th></tr>
  <tr><td>tconst</td><td>VARCHAR(12)</td><td>Identifiant de l'œuvre, clé primaire</td></tr>
  <tr><td>averageRating</td><td>DECIMAL(3,1)</td><td>Note moyenne</td></tr>
  <tr><td>numVotes</td><td>INT UNSIGNED</td><td>Nombre de votes</td></tr>
</table>

### Création de la base et des tables

Se connecter à mariaDB et executer:

```sql
CREATE DATABASE imdb
CHARACTER SET utf8mb4
COLLATE utf8mb4_unicode_ci;

USE imdb;
```

Création des tables:

```sql
CREATE TABLE title_basics (
    tconst VARCHAR(12) PRIMARY KEY,
    titleType VARCHAR(20),
    primaryTitle VARCHAR(512),
    originalTitle VARCHAR(512),
    isAdult TINYINT,
    startYear SMALLINT,
    endYear SMALLINT,
    runtimeMinutes INT,
    genres VARCHAR(128)
) ENGINE=InnoDB;
```

```sql
CREATE TABLE name_basics (
    nconst VARCHAR(12) PRIMARY KEY,
    primaryName VARCHAR(256),
    birthYear SMALLINT,
    deathYear SMALLINT,
    primaryProfession VARCHAR(255),
    knownForTitles TEXT
) ENGINE=InnoDB;
```
```sql
CREATE TABLE title_principals (
    tconst VARCHAR(12),
    ordering SMALLINT,
    nconst VARCHAR(12),
    category VARCHAR(32),
    job VARCHAR(512),
    characters TEXT,
    PRIMARY KEY (tconst, ordering),
    INDEX idx_principals_nconst (nconst)
) ENGINE=InnoDB;
```

```sql
CREATE TABLE title_ratings (
    tconst VARCHAR(12) PRIMARY KEY,
    averageRating DECIMAL(3,1),
    numVotes INT UNSIGNED
) ENGINE=InnoDB;
```


```sql
exit;
```
### Vérification du shéma

```
mariadb imdb
```

On regarde combien de tables existent:

```sql
SHOW TABLES;
```

On peut examiner leurs structures:

```sql
DESCRIBE title_basics;
DESCRIBE name_basics;
DESCRIBE title_principals;
DESCRIBE title_ratings;
```

### Préparation de l'importation

Nous allons maintenant décompresser les quatre archives.

```bash
mkdir -p /srv/imdb/data
cd /srv/imdb/download

for fichier in *.tsv.gz; do
    gzip -dk "$fichier"
done

mv *.tsv /srv/imdb/data/
```

On vérifie:

```bash
ls -lh /srv/imdb/data/
df -h /
```
<div class="alert alert-info">
Les fichiers décompressés représentent plusieurs gigaoctets. Il faut vérifier que l'espace disponible est suffisant avant de poursuivre.
</div>

### Configuration de la mémoire de InnoDB

Nous allons consacrer 4 Go au cache InnoDB. Cela devrait améliorer les performances des imports et des futures requêtes.

Crée un fichier de configuration dédié :

```bash
nano /etc/mysql/mariadb.conf.d/60-imdb.cnf
```

On y met:
```
[mysqld]
innodb_buffer_pool_size = 4G
```

On redémarre MariaDB
```bash
systemctl restart mariadb
```
### Reduction de la taille des données

Pour des raisons pédagogiques nous allons réduire les données, en effet l'importation des données complètes peut prendre plusieurs dizaines de minutes.

J'ai utilisé [ce script](script_reduction_imdb.py) qui permet de limiter le nombre d'oeuvres aux 20.000 plus récentes.

Vous trouverez les fichiers réduits **sur votre ordinateur** dans */partages/NSI/tp_imdb*.

A l'aide de la commande *scp*, copier ces 4 fichiers sur le serveur dans */srv/imdb/reduced*

### Importation des fichiers

Commençons par title_ratings, qui est le plus petit fichier. Cela nous permettra de valider notre méthode.

- Se connecter à la base imdb
- Executer l'import

```sql
LOAD DATA INFILE '/srv/imdb/reduced/title.ratings.tsv'
INTO TABLE title_ratings
CHARACTER SET utf8mb4
FIELDS TERMINATED BY '\t'
LINES TERMINATED BY '\n'
IGNORE 1 LINES;
```

Cette commande lit directement le fichier depuis le système de fichiers du serveur MariaDB.

Quelques points importants :

- INFILE : le fichier est lu par le serveur, pas par le client.
- CHARACTER SET utf8mb4 : précise l'encodage des données.
- IGNORE 1 LINES : ignore l'en-tête.
- FIELDS TERMINATED BY '\t' : les colonnes sont séparrées par des tabulations
- LINES TERMINATED BY '\n' : chaque ligne est séparée par un retour à la ligne
- IGNORE 1 LINES; la première ligne est ignorée.

On vérifie que l'importation a fonctionné:

```sql
SELECT COUNT(*) FROM title_ratings;
```

```sql
SELECT * FROM title_ratings LIMIT 10;
```
Importons les 3 autres tables

**Les oeuvres**

```sql
LOAD DATA INFILE '/srv/imdb/reduced/title.basics.tsv'
INTO TABLE title_basics
CHARACTER SET utf8mb4
FIELDS TERMINATED BY '\t'
LINES TERMINATED BY '\n'
IGNORE 1 LINES;
```

**Les personnes**

```sql
LOAD DATA INFILE '/srv/imdb/reduced/name.basics.tsv'
INTO TABLE name_basics
CHARACTER SET utf8mb4
FIELDS TERMINATED BY '\t'
LINES TERMINATED BY '\n'
IGNORE 1 LINES;
```


**Les participations**
```sql
LOAD DATA INFILE '/srv/imdb/reduced/title.principals.tsv'
INTO TABLE title_principals
CHARACTER SET utf8mb4
FIELDS TERMINATED BY '\t'
LINES TERMINATED BY '\n'
IGNORE 1 LINES;
```


## utilisation de DBeaver

[DBeaver](https://dbeaver.io/) est un outil qui permet de gérer énormément de bases de données différentes et de visualiser les tables et leurs relations.

Pendant l'importation des données nous allons utiliser ce logiciel.

Sur mariadb il faut créer un compte utilisateur dédié à dbeaver.

```sql
CREATE USER 'admin_tp'@'192.168.1.%'
IDENTIFIED BY 'dbeaver_mdp';

GRANT ALL PRIVILEGES ON imdb.* TO 'admin_tp'@'192.168.1.%';
```
Ce compte pourra administrer la base *imdb*, mais pas les éventuelles autres bases du serveur.


## Ajout des contraintes sur les clés etrangères

Voici la signification des tables:

- title_basics : Les films, séries et autres œuvres
- name_basics : Les personnes (acteurs, réalisateurs...)
- title_principals : Les personnes qui participent à une œuvre
- title_ratings : Les notes attribuées aux œuvres

Actuellement les clés étrangères n'ont pas été définies. Il en existe 3.

Examiner les tables et les données et essayer de les déterminer.

- Comment retrouver les informations d'un film à partir d'une participation ?
- Comment retrouver le nom d'un acteur à partir de sa participation ?
- À quel film correspond une note ?

Et voici la syntaxe générale pour ajouter une clé étrangère:

```sql
ALTER TABLE nom_de_la_table
ADD CONSTRAINT fk_nom_de_la_table
FOREIGN KEY (nom_de_la_colonne)
REFERENCES nom_de_l_autre_table(nom_de_la_colonne);
```

Vérifier avec dbeaver (couper la connexion et se reconnecter) que les clés étrangères sont maintenant visibles dans les tables.

## Premières requêtes 

Compléter les requetes qui permettent d'afficher les informations suivantes:

- [Les 250 films les mieux notés (qui ont eu + de 1000 votes)](https://www.imdb.com/fr/chart/top/).

```sql
SELECT
    title_basics.primaryTitle AS titre,
    title_basics.startYear AS annee,
    title_ratings.averageRating AS note,
    title_ratings.numVotes AS nombre_votes
FROM ...
JOIN ... ON ... = ...
WHERE title_basics.titleType = ...
  AND title_ratings.numVotes ...
ORDER BY ...
...
```

- [les 5 films d'horreur les mieux notés](https://www.imdb.com/fr/interest/horror/)

```sql
SELECT
    title_basics.primaryTitle AS titre,
    title_basics.startYear AS annee,
    ... AS genres,
    title_ratings.averageRating AS note,
    ... AS nombre_votes
FROM title_basics
JOIN ... ON ...
WHERE title_basics.titleType = 'movie'
  AND title_basics.genres ... '%Horror%'
  AND title_ratings.numVotes >= 1000
ORDER BY ...
...;
```

- les films dans lesquels Brad Pitt a été acteur du plus récent au plux vieux

```sql
SELECT
    title_basics.primaryTitle AS titre,
    title_basics.startYear AS annee,
    title_principals.category AS role,
    title_principals.characters AS personnages
FROM name_basics
JOIN ...
JOIN ...
WHERE name_basics.primaryName = 'Brad Pitt'
ORDER BY ...;
```

Pour cette dernière requête, observez le temps de la requête.
Ce temps vient du fait de l'obligation de parcourir toutes les personnes à la rechercher de 'Brad Pitt'.

Pour améliorer cela nous allons ajouter un **index** sur le nom de l'acteur.
Cet index permettra de retrouver rapidement la ou les lignes correspondant à Brad Pitt.

```sql
CREATE INDEX idx_name_primaryName
ON name_basics(primaryName);
```

Exécutez à nouveau la requête et observez la différence.


Voici les 1000 opremières entrées de l'index

```sql
SELECT primaryName, nconst
FROM name_basics FORCE INDEX (idx_name_primaryName)
ORDER BY primaryName
LIMIT 1000;
```

## Gestion des utilisateurs et des privilèges
